# Batch Inference Pipeline

This notebook demonstrates how to run the batch inference pipeline over multiple
reflectometry experiments and how to analyse and visualise the results.

Topics covered:

1. Running a small batch via `BatchInferencePipeline`
2. Loading saved batch results from JSON
3. Summary statistics and MAPE distribution
4. Per-parameter MAPE breakdown plot
5. Regenerating plots from existing results with `replot_batch_results`

## 1. Setup

In [ ]:
import sys
import os
import json
from pathlib import Path

# Add project root to path when running from notebooks/ subdirectory
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Change working directory so relative paths in pipeline resolve correctly
os.chdir(PROJECT_ROOT)

import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

# Disable TeX rendering - paper.mplstyle enables it but TeX fonts may not be installed
plt.rcParams["text.usetex"] = False

from config import DATA_DIRECTORY, BATCH_RESULTS_DIR

print(f"Project root      : {PROJECT_ROOT}")
print(f"Data directory    : {DATA_DIRECTORY}")
print(f"Batch results dir : {BATCH_RESULTS_DIR}")

## 2. Run a Small Batch

`BatchInferencePipeline` orchestrates the full workflow: experiment discovery,
preprocessing, inference, metrics, and result serialisation.

Set `NUM_EXPERIMENTS` to a small number (e.g. 5) for a quick demonstration run.
Set to `None` to process all available experiments in the layer split.

In [ ]:
from batch_pipeline import BatchInferencePipeline

NUM_EXPERIMENTS  = 30      # set to None to process all
LAYER_COUNT      = 1
PRIORS_DEV       = 30     # % of constraint span (for constraint_based priors)
SLD_MODE         = "none" # "none" | "backing" | "all"
PROMINENT        = False  # True = restrict to prominent-feature subset

pipeline = BatchInferencePipeline(
    num_experiments=NUM_EXPERIMENTS,
    layer_count=LAYER_COUNT,
    data_directory=Path(PROJECT_ROOT) / DATA_DIRECTORY,
    output_dir=Path(PROJECT_ROOT) / BATCH_RESULTS_DIR,
    priors_type="constraint_based",
    narrow_priors_deviation=PRIORS_DEV / 100.0,
    fix_sld_mode=SLD_MODE,
    use_prominent_features=PROMINENT,
    inference_backend="nf",
    config_name="example_nf_config_reflectorch.yaml",
)

# Path debugging
print(f"project_root: {PROJECT_ROOT}")
print(f"  data_directory: {Path(PROJECT_ROOT) / DATA_DIRECTORY}")
print(f"  output_dir: {Path(PROJECT_ROOT) / BATCH_RESULTS_DIR}")

batch_results = pipeline.run()
output_dir = pipeline.output_dir
print(f"\nResults saved to: {output_dir}")

## 3. Load Results from an Existing Batch

To work with a previously completed run without re-running inference, load its `batch_results.json` directly.
Update `EXISTING_BATCH_DIR` to point at any directory under `batch_inference_results/`.

In [ ]:
# Derived automatically from the batch run above
EXISTING_BATCH_DIR = Path(PROJECT_ROOT) / output_dir

results_file = EXISTING_BATCH_DIR / "batch_results.json"

with open(results_file) as f:
    loaded_results = json.load(f)

successful = {k: v for k, v in loaded_results.items() if v.get("success", False)}
print(f"Loaded {len(loaded_results)} total results, {len(successful)} successful")

## 4. Summary Statistics

`batch_analysis.create_summary_statistics` aggregates per-experiment metrics into dataset-level statistics.

In [ ]:
from batch_analysis import create_summary_statistics, print_summary_statistics, print_mape_distribution

summary = create_summary_statistics(successful, layer_count=LAYER_COUNT)
print_summary_statistics(summary)
print()
print_mape_distribution(successful)

## 5. MAPE Distribution and Parameter Breakdown Plots

`create_batch_analysis_plots` produces all standard batch plots in one call.
Set `save=True` to write PDFs to `output_dir`.

In [ ]:
from plotting_utils import create_batch_analysis_plots
%matplotlib inline


plot_paths = create_batch_analysis_plots(
    successful,
    layer_count=LAYER_COUNT,
    output_dir=EXISTING_BATCH_DIR,
    save=False,
)

for name, path in plot_paths.items():
    if path is not None:
        print(f"  {name}: {path}")

## 6. Re-Generate Plots from CLI

Plots can be regenerated from any completed batch directory without re-running inference:

```bash
python replot_batch_results.py --results-dir batch_inference_results/<batch_dir>
```

Or using the Python API:

In [ ]:
from replot_batch_results import replot_batch_results

saved_plots = replot_batch_results(
    results_dir=str(EXISTING_BATCH_DIR),
    output_dir=str(EXISTING_BATCH_DIR / "replot_nb"),
)
print(f"Regenerated {len(saved_plots)} plots.")